<h1>Extracting Additional Metadata Associated with SORD</h1>

In this notebook, we provide queries/code snippets used to extract FilteredBadges, FilteredUsers, FilteredVotes and FilteredTags. 

<h2>1. FilteredUsers</h2>

The following SQL query filters users who have asked /answered or commented recommendation related content from the Stack Overflow data dump.

<h2>2. FilteredBadges</h2>

The following SQL query filters badges attained by users filtered above (users who have asked /answered or commented recommendation related content).

<h2 style="color: red;">3. FilteredPosts</h2>

We combined all posts (questions) that were included in the following:
<ol>
<li>Questions_Title_Contain, Questions_Title_LikeMinusContain, Questions_Body_Contain, Questions_Body_LikeMinusContain (through Id)</li>
<li>Answers_Contain, Answers_LikeMinusContain (through ParentId)</li> 
<li>Comments_Contain, Comments_LikeMinusContain (through PostId)</li>
</ol>
The following script was used for this purpose.

<p style="color:red;">We extracted FilteredPosts to generate FilteredVotes and FilteredTags. However, we have not published FilteredPosts in the dataset due to its large size. Those who wish to utilize FilteredPosts can use the data dump parsers and above mentioned query to generate it themselves or can simply combine the PostIds from the union of above mentioned tables and get those posts from the Stack Exchange Data Explorer (SEDE). </p>

<h2>4. FilteredVotes</h2>

<h2>5. FilteredTags</h2>

<h3>Why FilteredTags is important?</h3>

By extracting tags used in SORD (directly in questions that are included in SORD or indirectly the parent question (post) of answers/comments included in SORD), we get an idea about which topics, tools, technologies, concepts or software is potentially being recommended by developers in their discussions. This may help community further filter/refine particular types of tags and their assocaited content from SORD to develop specific type of recommendations systems e.g. a recommender to recommend programming lanaugages for particular applications or use cases. 

We used the following code snippet to extract all the unique tags associated with the stack overflow questions (posts) that had one or more recommendation related keywords in their title, body, answers or comments.

<h4>Extracting Unique Tags with Frequency of Occurence in SORD</h4>

As tags are associated with questions only (not with answers or comments), having FilteredPosts was necessary. Once we have all posts included in SORD, we simply extracted the tags column from FilteredPosts and identified the unique set of tags.

In [3]:
import pyodbc
import re
import csv
from collections import defaultdict

# =========================
# SQL Server connection
# =========================
conn_str = (
    "DRIVER={SQL Server};"
    "SERVER=DESKTOP-EQTUGG1;"
    "DATABASE=SORecommendationsDb;"
    "Trusted_Connection=yes;"  # or UID/PWD
)

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

# =========================
# Fetch Name column
# =========================
query = "SELECT Tags FROM [SORecommendationsDb].[dbo].[FilteredPosts]"
cursor.execute(query)

pattern = re.compile(r"<([^>]+)>")

# Dictionary: label -> row count
label_counts = defaultdict(int)

# =========================
# Process rows
# =========================
for row in cursor:
    name_value = row[0]

    if not name_value:
        continue

    # Extract labels from the row
    labels = pattern.findall(name_value)

    # Count each label only once per row
    unique_labels_in_row = set(label.strip() for label in labels)

    for label in unique_labels_in_row:
        label_counts[label] += 1

# =========================
# Sort alphabetically
# =========================
sorted_labels = sorted(label_counts.items(), key=lambda x: x[0].lower())

# =========================
# Write to CSV
# =========================
output_file = "FilteredTags.csv"

with open(output_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Tag", "Count"])
    for label, count in sorted_labels:
        writer.writerow([label, count])

# Cleanup
cursor.close()
conn.close()

print(f"CSV file created: {output_file}")


CSV file created: FilteredTags.csv
